# OpenCV

In this lab, we'll use advanced OpenCV tools.

Of course, OpenCV offers all the usual image processing tools, but to avoid overlapping with Pillow or Scikit-image, we'll focus on the models it provides for face detection, as well as video processing and object tracking.

## IMPORTANT
Before starting this lab, run the code cell below without looking at its content. It fetches the data and applies a few transformations so the lab runs properly on Colab

In [ ]:
# @title Run me: environment setup
import glob
import os
import pathlib
from base64 import b64encode

import cv2
from IPython.display import HTML, clear_output

if not pathlib.Path("polo.mp4").exists():
  !pip --quiet uninstall --yes opencv-contrib-python opencv-python opencv-contrib-python-headless && pip install --quiet opencv-contrib-python
  !wget -O polo.zip https://data.votchallenge.net/sequences/acc3aeade6fb699adb7ca1c28866dd00f06d8fcd02c4cff17539a3c9d718d38976f637f4499a8763ae65f93d71fcd37494747bec7bbb8955a469b9f3fd9783aa.zip
  !unzip polo.zip -d polo
  !wget https://github.com/opencv/opencv_extra/raw/c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking/goturn.caffemodel.zip.001
  !wget https://github.com/opencv/opencv_extra/raw/c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking/goturn.caffemodel.zip.002
  !wget https://github.com/opencv/opencv_extra/raw/c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking/goturn.caffemodel.zip.003
  !wget https://github.com/opencv/opencv_extra/raw/c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking/goturn.caffemodel.zip.004
  !wget https://github.com/opencv/opencv_extra/raw/c4219d5eb3105ed8e634278fad312a1a8d2c182d/testdata/tracking/goturn.prototxt
  !cat goturn.caffemodel.zip.0* > ./goturn.caffemodel.zip
  !unzip goturn.caffemodel.zip
  !wget https://github.com/ultralytics/yolov5/releases/download/v6.0/yolov5s.pt
  !pip install yolov5 fastapi kaleido python-multipart uvicorn
  !wget https://ndownloader.figshare.com/files/5976018 -O lfw.tgz
  !tar -xf lfw.tgz
  !rm lfw.tgz

  video_name = "polo.mp4"
  images = sorted(glob.glob("polo/*.jpg"))
  frame = cv2.imread(images[0])
  height, width, layers = frame.shape
  video = cv2.VideoWriter(
    video_name, cv2.VideoWriter_fourcc(*"mp4v"), 10, (width, height)
  )
  for image in images:
    video.write(cv2.imread(image))
  video.release()

  !rm -rf polo
  !rm *.zip*


def show_video(video_path: str):
  # Compressed video path
  compressed_path = "displayed_video.mp4"

  os.system(f"ffmpeg -y -i {video_path}  -vcodec libx264 {compressed_path}")

  # Show video
  mp4 = pathlib.Path(compressed_path).open("rb").read()
  data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
  display(
    HTML(
      """
  <video width=640 controls>
        <source src="%s" type="video/mp4">
  </video>
  """
      % data_url
    )
  )


clear_output()

## Imports

In [ ]:
import glob
import os
from base64 import b64encode
from collections.abc import Callable

import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import yolov5
from google.colab.patches import cv2_imshow
from IPython.display import HTML, clear_output
from tqdm import tqdm

## Face Detection

In this section, you'll set up a face detection process.
From the image data in the `lfw` folder, pick one or more images and detect the face and the eyes in rectangles, then display the image with the detections.

The OpenCV documentation is unfortunately patchy, not to say missing, especially for Python. Here's what you'll need to do this exercise step by step:
 * Load an image with `cv2.imread`
 * The detection tool is `cv2.CascadeClassifier(model_param_path:str)`. It takes a path to the model parameters. You'll need two parameter files, one for the face and one for the eyes, which you can get respectively with the following code:
   * `cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'`
   * `cv2.data.haarcascades + 'haarcascade_eye.xml'`
 * The detector works on black and white images
 * To run the detector on an image, you'll need to call the `detectMultiScale` method. You'll find a detailed explanation of its parameters on [`stackoverflow`](https://stackoverflow.com/questions/36218385/parameters-of-detectmultiscale-in-opencv-using-python)
 * The method returns a list of boxes matching the requested detection
 * Draw the detection result on the image with [`cv2.rectangle`](https://docs.opencv.org/3.1.0/d6/d6e/group__imgproc__draw.html#ga07d2f74cadcf8e305e810ce8eed13bc9) (if you're lost --> explanation on [`stackoverflow`](https://stackoverflow.com/questions/23720875/opencv-draw-a-rectangle-around-a-region))
 * You can display your image with the method of your choice or use the `cv2_imshow` method (ask your kind trainer for an explanation)

_Good luck_

In [ ]:
# Your code here

### Solution

In [ ]:
# Load the cascade
face_cascade = cv2.CascadeClassifier(
  cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")

# Read the input image
img = cv2.imread("lfw/Adam_Rich/Adam_Rich_0001.jpg")

# Convert to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Detect the faces
faces = face_cascade.detectMultiScale(gray, 1.2, 5)
eyes = eye_cascade.detectMultiScale(gray, 1.2, 5)

# Draw a rectangle around the faces
for x, y, w, h in faces:
  cv2.rectangle(img, (x, y), (x + w, y + h), color=(0, 255, 0), thickness=2)
for x, y, w, h in eyes:
  cv2.rectangle(img, (x, y), (x + w, y + h), color=(255, 0, 0), thickness=2)

# Display the output
cv2_imshow(img)

## Video Processing

 ### Displaying a Video
 You'll now process a video by tracking an object or a person in it.

 You'll find a file named `polo.mp4` in your folder. Your kind trainer has provided a display function to view it directly in Colab. In the next code cell, use the `show_video(video_path:str)` function

In [ ]:
# Your code here

#### Solution

In [ ]:
show_video("polo.mp4")

### Defining the Capture Box
To initialize the tracking algorithm, you'll need to initialize it on the first frame of the video, providing the area of the object to follow.

Use `cv2.VideoCapture` to read the video frame by frame with its `read` function. Display the result to help you define the capture area on the object of your choice.

In [ ]:
# Your code here

#### Solution

In [ ]:
def plot_im_with_rect(image, rect):
  fig, ax = plt.subplots()
  ax.imshow(image)

  rect = patches.Rectangle(
    rect[:2], rect[2], rect[3], linewidth=1, edgecolor="r", facecolor="none"
  )
  ax.add_patch(rect)
  display(fig)


def plot_im_with_rects(image, rects):
  im = image.copy()
  # Draw rectangles on the original image
  for x, y, w, h in rects:
    cv2.rectangle(im, (x, y), (x + w, y + h), (0, 0, 255), 2)

  # Display the image.
  cv2_imshow(im)


HORSE_RECT = (240, 240, 120, 110)

video = cv2.VideoCapture("polo.mp4")
ret, frame = video.read()
plot_im_with_rects(frame, [HORSE_RECT])

### Tracking
Write a tracking function:

```python
def tracking(video_path:str, # Path of the video to process
             init_rect:Tuple[int], # Initial detection rectangle
             model_create:Callable # Model used for the capture
             )->List[np.ndarray]: # Returns the list of frames after detection
```
The `model_create` parameter is a function that creates the tracking model you want to use. Here's the exhaustive list:
* BOOSTING: `cv2.legacy.TrackerBoosting_create()`
* MIL: `cv2.TrackerMIL_create() `
* KCF: `cv2.TrackerKCF_create() `
* TLD: `cv2.legacy.TrackerTLD_create()`
* MEDIANFLOW: `cv2.legacy.TrackerMedianFlow_create()`
* GOTURN: `cv2.TrackerGOTURN_create()`
* MOSSE: `cv2.legacy.TrackerMOSSE_create()`
* CSRT: `cv2.TrackerCSRT_create()`

Pick the one of your choice and produce, for each frame, a new frame with the tracking box ([Explanation of the pros and cons of each model](https://learnopencv.com/object-tracking-using-opencv-cpp-python/)).

The tracker is used as follows:
 * Create the model `tracker = model_create()`
 * Initialize the tracker with the `init` function: `tracker.init(frame, init_rect)`
 * You can now call `tracker.update(frame)` to run the detection. The function returns two values:
  * A boolean telling whether the detection worked
  * The detection box for the given image

**WARNING: the tracker needs boxes in xywh format, i.e. the coordinates of one of the rectangle's corners (xy) and the size of the box (wh)**

*Example:
xyxy format (100,100,140,150) <-> xywh format (100,100,40,50)*

In [ ]:
# Your code here

#### Solution

In [ ]:
def tracking(
  video_path: str,
  init_rect: tuple[int],
  model_create: Callable = cv2.TrackerGOTURN_create,
) -> list[np.ndarray]:
  frames_list = []
  video = cv2.VideoCapture("polo.mp4")
  ret, frame = video.read()
  if ret:
    tracker = model_create()
    tracker.init(frame, init_rect)
    while ret:
      ok, box = tracker.update(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
      (x, y, w, h) = (int(v) for v in box)
      if ok:
        # Tracking success
        cv2.rectangle(frame, (x, y), (x + w, y + h), color=(0, 0, 255), thickness=2)
        frames_list.append(frame)

      ret, frame = video.read()
  return frames_list

### Turning a List of Frames Into a Video
You should now have a list of frames labeled with the tracking boxes. Convert the list of frames into a video.

To do so, use `cv2.VideoWriter`, which works a bit like `cv2.VideoCapture`, but for writing.
```python
cv2.VideoWriter(video_name, # Path of the video to create
                cv2.VideoWriter_fourcc(*'mp4v'), # Codec selection
                5, # Framerate
                (width,height) # Resolution
                )
```
Then use the `write(frame)` function on each frame and, at the very end of the process, `video.release()` to free the memory.

Finally, display your video.

*Bonus information:*

*Fourcc (Four character code) is a 4-character naming convention for video codecs. For compatibility with Google Colab, keep `mp4v`, but here's the [`exhaustive list`](http://abcavi.kibi.ru/fourcc.php) for your curiosity.*

In [ ]:
# Your code here

#### Solution

In [ ]:
def convert_to_video(frames_list, video_name):
  height, width, _ = frames_list[0].shape
  video = cv2.VideoWriter(
    video_name, cv2.VideoWriter_fourcc(*"mp4v"), 10, (width, height)
  )

  for frame in tqdm(frames_list):
    video.write(frame)

  video.release()

In [ ]:
models = {
  "BOOSTING": cv2.legacy.TrackerBoosting_create,
  "MIL": cv2.TrackerMIL_create,
  "KCF": cv2.TrackerKCF_create,
  "TLD": cv2.legacy.TrackerTLD_create,
  "MEDIANLOW": cv2.legacy.TrackerMedianFlow_create,
  "GOTURN": cv2.TrackerGOTURN_create,
  "MOSSE": cv2.legacy.TrackerMOSSE_create,
  "CSRT": cv2.TrackerCSRT_create,
}
for model_name in models:
  print(model_name)
  frames_list = tracking("polo.mp4", HORSE_RECT, models[model_name])
  convert_to_video(frames_list, "polo_capture.mp4")
  show_video("polo_capture.mp4")

## Video Tracking With YOLO
We'll now track with a state-of-the-art model outside OpenCV, YOLO, available in the yolov5 library, to compare.

YOLO works a bit differently. You don't need to initialize it, and it's used as follows:
```python
yolo = yolov5.load('yolov5s.pt') # Loading
...
detections = yolo(frame)
detections.xyxy # The list of detection boxes, of size (nb_boxes, 6)
```
 * The first line loads the model parameters. They've already been downloaded for you into the `yolov5s.pt` file
 * `detections.xyxy` contains the detection boxes. It has 6 columns; the first 4 are the box coordinates in xyxy format

In [ ]:
# Your code here

### Solution

In [ ]:
def yolo_tracking(video_path: str) -> list[np.ndarray]:
  frames_list = []
  model = yolov5.load("yolov5s.pt")
  video = cv2.VideoCapture(video_path)
  ret, frame = video.read()
  while ret:
    detections = model(frame)

    boxes = np.array(detections.xyxy[0][:, :4]).astype(int)

    for x1, y1, x2, y2 in boxes:
      cv2.rectangle(frame, (x1, y1), (x2, y2), color=(0, 255, 0), thickness=2)
    frames_list.append(frame)
    ret, frame = video.read()
  return frames_list

In [ ]:
frames_list = yolo_tracking("polo.mp4")
convert_to_video(frames_list, "test_capture.mp4")
show_video("test_capture.mp4")